# 可选实验: 成本函数（Cost Function ）
<figure>
    <center> <img src="../work/images/C1_W1_L3_S2_Lecture_b.png"  style="width:1000px;height:200px;" ></center>
</figure>

## 目标
在本次实验中：
- 你将实现并探索单变量线性回归的代价函数（Cost Function）。


## 工具
在本次实验中，你将使用：
- NumPy, 一个广受欢迎的科学计算专用库.
- Matplotlib, 一个用于绘制数据的常用工具库.
- 在本地目录下的“lab_utils_uni.py”文件中的本地绘图程序。

In [ ]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from lab_utils_uni import plt_intuition, plt_stationary, plt_update_onclick, soup_bowl
plt.style.use('./deeplearning.mplstyle')

## 问题陈述

您希望有一个模型，能够根据房屋的面积来预测其价格。 让我们使用与上一次实验相同的两个数据点——面积为 1000 平方英尺的房屋售价为 \\$300,000 美元，面积为 2000 平方英尺的房屋售价为 \\$500,000 美元。


| Size (1000 sqft)     | Price (1000s of dollars) |
| -------------------| ------------------------ |
| 1                 | 300                      |
| 2                  | 500                      |


In [ ]:
x_train = np.array([1.0, 2.0])               #(以1000平方英尺为单位的面积)
y_train = np.array([300.0, 500.0])           #(以1000美元为单位的价格)

## 计算成本
本作业中的‘代价（cost）’一词可能会让人有些困惑，因为数据本身就是关于住房成本（housing cost）的。在这里，‘代价’是衡量模型对房屋目标价格（target price）预测好坏的一个指标。而‘价格（price）’一词则是专门用于指代住房数据本身。


单变量代价函数的方程为:
  $$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2 \tag{1}$$ 
 
而 
  $$f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{2}$$
  
- $f_{w,b}(x^{(i)})$ 模型在由参数 $w, b$ 确定的情况下，对样本 $x^{(i)}$ 的评估结果（预测值）.  
- $(f_{w,b}(x^{(i)}) -y^{(i)})^2$ 是目标值（真实值）与预测值之间的误差平方（平方差）. 
- 将所有 $m$ 个样本的这些误差平方相加，并除以 $2m$，从而得出代价函数 $J(w,b)$ .
>请注意，在讲解总结中，范围通常是从 1 到 m，而代码中则是从 0 到 m - 1.


下面的代码通过遍历每个示例来计算成本。在每次循环中：
- `f_wb`, 一个预测值已被计算出来.
- 将目标值与预测值之间的差值计算出来，并将其平方.
- 以上计算结果被加到了cost_sum中.

In [ ]:
def compute_cost(x, y, w, b): 
    """
    Computes the cost function for linear regression.
    
    Args:
      x (ndarray (m,)): Data, m examples 
      y (ndarray (m,)): target values
      w,b (scalar)    : model parameters  
    
    Returns
        total_cost (float): The cost of using w,b as the parameters for linear regression
               to fit the data points in x and y
    """
    # number of training examples
    m = x.shape[0] 
    
    cost_sum = 0 
    for i in range(m): 
        f_wb = w * x[i] + b   
        cost = (f_wb - y[i]) ** 2  
        cost_sum = cost_sum + cost  
    total_cost = (1 / (2 * m)) * cost_sum  

    return total_cost

## 成本函数直观理解



<img align="left" src="../work/images/C1_W1_Lab02_GoalOfRegression.PNG"    style=" width:40%; padding: 10px;  " /> 
您的目标是找到一个模型 $f_{w,b}(x) = wx + b$, 其参数为 $w,b$,  能够在给定输入 $x$ 的情况下准确预测房屋价值. 成本是衡量模型在训练数据上准确性的一个指标.



上面的成本方程 (1) 表明，如果可以选取 $w$ 和 $b$ 使得预测值 $f_{w,b}(x)$ 与目标数据 $y$ 相匹配, 那么 $(f_{w,b}(x^{(i)}) - y^{(i)})^2 $ 这一项将为零， 从而使成本最小化。在这个简单的两点示例中，您完全可以实现这一点！


在之前的实验中, 您已经确定 $b=100$ 是一个最优解，因此让我们将 $b$ 设为 100，并专注于调整 $w$.

<br/>
在下方，使用滑块控件来选择使成本最小化的 $w$ 值，图表可能需要几秒钟才能更新.

In [ ]:
plt_intuition(x_train,y_train)

图表中有几个值得注意的要点.
- 当 $w = 200$ 时, 成本最小化，这与之前实验的结果一致。
- 由于成本方程中目标值与预测值之间的差值被平方， 当 $w$ 过大或过小时，成本会迅速增加。
- 使用通过最小化成本选出的 `w` 和 `b` 会得到一条与数据完美拟合的直线。

## 成本函数可视化 - 3D

您可以通过绘制3D图或使用等高线图来查看成本如何随 `w` 和 `b` 两者变化.   
值得注意的是，本课程中的一些绘图可能会变得相当复杂。绘图例程已提供，虽然阅读代码以熟悉这些方法可能具有指导意义，但成功完成本课程并不需要这样做。这些例程位于本地目录下的lab_utils_uni.py 文件中。

### 更大的数据集
观察一个包含更多数据点的场景是具有指导意义的。这个数据集包含的数据点并不在同一条直线上。这对成本方程意味着什么？我们能否找到 $w$ 和 $b$ 使成本为 0？

In [ ]:
x_train = np.array([1.0, 1.7, 2.0, 2.5, 3.0, 3.2])
y_train = np.array([250, 300, 480, 430, 630, 730,])

在等高线图中，点击一个点来选择 `w` 和 `b` 以实现最低成本。 使用等高线来指导您的选择。请注意，图表可能需要几秒钟才能更新。

In [ ]:
plt.close('all') 
fig, ax, dyn_items = plt_stationary(x_train, y_train)
updater = plt_update_onclick(fig, ax, x_train, y_train, dyn_items)

请注意上图（左侧图表）中的虚线。它们代表了训练集中每个样本对总成本的贡献部分。在这种情况下，大约 $w=209$ 和 $b=2.4$ 的值能提供较低的成本。   

请注意，因为我们的训练样本不在一条直线上，所以最小成本不为零。

### 凸成本曲面

成本函数对损失进行平方这一事实，确保了“误差曲面”是凸的，就像一个汤碗。它总会有一个最小值，可以通过沿着所有维度的梯度方向到达。在前面的图中，由于 $w$ 和 $b$ 维度的尺度不同，这一点不容易识别。下面的图展示了 $w$ 和 $b$ 对称的情况，这在讲座中展示过。​

In [ ]:
soup_bowl()

# 恭喜!
您已经学到了以下内容：
 - 成本函数提供了一个衡量您的预测与训练数据匹配程度的指标。
 - 最小化成本可以提供 $w$ 和 $b$ 的最优值。